# DeHate TikTok dataset

This notebook analyses the public `DeHate.xlsx` file. The full DeHate corpus mixes TikTok and BitChute, so all TikTok-specific findings are calculated from the TikTok subset only.

Sources: [official repository](https://github.com/Multimodal-Intelligence-Lab-MIL/DeHate) and [paper record](https://doi.org/10.1145/3746027.3758272).

## 1. Load and verify the local official file

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

ROOT = Path.cwd()
if not (ROOT / 'downloaded_datasets').exists() and (ROOT.parent / 'downloaded_datasets').exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'downloaded_datasets' / 'dehate'
XLSX_PATH = DATA_DIR / 'DeHate.xlsx'
if not XLSX_PATH.exists():
    raise FileNotFoundError(XLSX_PATH)

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

display(pd.DataFrame([{
    'file': XLSX_PATH.name,
    'bytes': XLSX_PATH.stat().st_size,
    'MiB': XLSX_PATH.stat().st_size / 1024**2,
    'sha256': sha256(XLSX_PATH),
}]).round({'MiB': 3}))

data = pd.read_excel(XLSX_PATH, dtype={'Video ID': 'string'})
print('Shape:', data.shape)
display(data.head(3))

,file,bytes,MiB,sha256
0,DeHate.xlsx,1267040,1.21,581bd48a4be39cb9a300b81c24788aad57a1539412d730...


Shape: (6689, 17)


,Video ID,Hate,Explicit or Implicit,Hate Segment,Textual Content,Visual Content,Audio Content,Race,Gender,Religion,Sexual Orientation,Disability,Immigration,Platform,title,desc,Split
0,sMSxeiqh1Y3T,0,NonHate,[],0,0,0,0,0,0,0,0,0,BitChute,"DON'T KID YOURSELF. THEY'RE ALL IN ON IT, AND ...","To survive what is coming, read/study: https:/...",train
1,Lt6gALRyF3yF,0,NonHate,[],0,0,0,0,0,0,0,0,0,BitChute,NYPD 10TH PRECINCT TYRANTS 🚨,Ely on YouTube who's a #1A auditor gets his ri...,train
2,7168209157156343046,0,NonHate,[],0,0,0,0,0,0,0,0,0,TikTok,/,#duet with @keir #visitaustralia Then fkn spid...,train


## 2. Structure and comment availability

In [2]:
structure = pd.DataFrame([
    {'content': 'Platform and video ID', 'available': 'Yes', 'columns': 'Platform, Video ID'},
    {'content': 'Hate label', 'available': 'Yes', 'columns': 'Hate, Explicit or Implicit'},
    {'content': 'Segment-level localisation', 'available': 'Yes', 'columns': 'Hate Segment'},
    {'content': 'Contributing modalities', 'available': 'Yes', 'columns': 'Textual, Visual, Audio Content'},
    {'content': 'Target-group labels', 'available': 'Yes', 'columns': 'Race through Immigration'},
    {'content': 'Title and description', 'available': 'Yes', 'columns': 'title, desc'},
    {'content': 'Train/validation/test split', 'available': 'Yes', 'columns': 'Split'},
    {'content': 'Comment count, comment rows, or comment text', 'available': 'No', 'columns': 'not included'},
    {'content': 'Raw videos/audio', 'available': 'Application only', 'columns': 'not in public XLSX'},
])
display(structure)

comment_columns = [column for column in data.columns if 'comment' in column.lower()]
print('Comment-related columns:', comment_columns)
print('Mean/median comments per video can be calculated:', bool(comment_columns))

,content,available,columns
0,Platform and video ID,Yes,"Platform, Video ID"
1,Hate label,Yes,"Hate, Explicit or Implicit"
2,Segment-level localisation,Yes,Hate Segment
3,Contributing modalities,Yes,"Textual, Visual, Audio Content"
4,Target-group labels,Yes,Race through Immigration
5,Title and description,Yes,"title, desc"
6,Train/validation/test split,Yes,Split
7,"Comment count, comment rows, or comment text",No,not included
8,Raw videos/audio,Application only,not in public XLSX


Comment-related columns: []
Mean/median comments per video can be calculated: False


## 3. Check the published counts

In [3]:
count_checks = pd.DataFrame([
    {'measure': 'All videos', 'published': 6689, 'downloaded': len(data)},
    {'measure': 'TikTok videos', 'published': 1637, 'downloaded': int(data['Platform'].eq('TikTok').sum())},
    {'measure': 'BitChute videos', 'published': 5052, 'downloaded': int(data['Platform'].eq('BitChute').sum())},
    {'measure': 'Unique video IDs', 'published': 6689, 'downloaded': data['Video ID'].nunique()},
])
count_checks['difference'] = count_checks['downloaded'] - count_checks['published']
count_checks['identical'] = count_checks['difference'].eq(0)
display(count_checks)

platform_labels = data.groupby(['Platform', 'Explicit or Implicit']).size().unstack(fill_value=0)
display(platform_labels)

,measure,published,downloaded,difference,identical
0,All videos,6689,6689,0,True
1,TikTok videos,1637,1637,0,True
2,BitChute videos,5052,5052,0,True
3,Unique video IDs,6689,6689,0,True


Explicit or Implicit,Explicit,Implicit,NonHate
Platform,,,
BitChute,1055,835,3162
TikTok,115,115,1407


## 4. TikTok-only hate-label distribution

In [4]:
tiktok = data[data['Platform'].eq('TikTok')].copy()
label_distribution = tiktok['Explicit or Implicit'].value_counts().rename('videos').to_frame()
label_distribution['percent'] = label_distribution['videos'] / len(tiktok) * 100
display(label_distribution.round(2))

split_table = pd.crosstab(tiktok['Split'], tiktok['Explicit or Implicit'], margins=True)
display(split_table)

,videos,percent
Explicit or Implicit,,
NonHate,1407,85.95
Implicit,115,7.03
Explicit,115,7.03


Explicit or Implicit,Explicit,Implicit,NonHate,All
Split,,,,
test,31,26,274,331
train,76,79,985,1140
val,8,10,148,166
All,115,115,1407,1637


## 5. Interesting point: how hate is expressed across modalities

In [5]:
modality_columns = ['Textual Content', 'Visual Content', 'Audio Content']

def modality_name(row):
    names = [name.replace(' Content', '') for name in modality_columns if row[name] == 1]
    return ' + '.join(names) if names else 'None (non-hate)'

tiktok['modality_combination'] = tiktok.apply(modality_name, axis=1)
modality_table = pd.crosstab(tiktok['modality_combination'], tiktok['Explicit or Implicit'])
modality_table['All'] = modality_table.sum(axis=1)
display(modality_table.sort_values('All', ascending=False))

tiktok_hate = tiktok[tiktok['Hate'].eq(1)]
print('TikTok hateful videos:', len(tiktok_hate))
print('Hateful TikTok videos involving audio:', int(tiktok_hate['Audio Content'].eq(1).sum()))
print('Audio share among hateful TikTok videos:', f"{tiktok_hate['Audio Content'].eq(1).mean() * 100:.2f}%")

Explicit or Implicit,Explicit,Implicit,NonHate,All
modality_combination,,,,
None (non-hate),0,0,1407,1407
Visual + Audio,50,42,0,92
Audio,35,31,0,66
Textual + Visual + Audio,17,12,0,29
Textual,10,13,0,23
Visual,3,12,0,15
Textual + Audio,0,3,0,3
Textual + Visual,0,2,0,2


TikTok hateful videos: 230
Hateful TikTok videos involving audio: 190
Audio share among hateful TikTok videos: 82.61%


## 6. Interesting point: targeted groups and multi-label annotations

In [6]:
target_columns = ['Race', 'Gender', 'Religion', 'Sexual Orientation', 'Disability', 'Immigration']
target_counts = tiktok_hate[target_columns].sum().sort_values(ascending=False).rename('label_occurrences').to_frame()
target_counts['share_of_hateful_videos'] = target_counts['label_occurrences'] / len(tiktok_hate) * 100
display(target_counts.round(2))

targets_per_video = tiktok_hate[target_columns].sum(axis=1)
display(targets_per_video.value_counts().sort_index().rename('videos').to_frame())
print('Target labels may overlap, so column totals can exceed the number of hateful videos.')

,label_occurrences,share_of_hateful_videos
Race,97,42.17
Immigration,51,22.17
Religion,37,16.09
Disability,33,14.35
Sexual Orientation,26,11.30
Gender,25,10.87


,videos
1,198
2,28
3,3
6,1


Target labels may overlap, so column totals can exceed the number of hateful videos.


## 7. Text fields and missing values

In [7]:
tiktok['description_characters'] = tiktok['desc'].fillna('').str.len()
display(tiktok['description_characters'].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).to_frame())

missing = pd.DataFrame({
    'missing_count': tiktok.isna().sum(),
    'missing_percent': tiktok.isna().mean() * 100,
}).query('missing_count > 0').sort_values('missing_count', ascending=False)
display(missing.round(2))

,description_characters
count,"1,637.00"
mean,112.07
std,163.63
min,1.00
25%,40.00
50%,82.00
75%,133.00
90%,211.40
95%,297.40
99%,803.04


,missing_count,missing_percent


# Findings

- The public workbook contains **6,689 unique videos**, and its platform and label counts exactly match the repository: 1,637 TikTok videos and 5,052 BitChute videos.
- The TikTok subset has 230 hateful videos: 115 explicit and 115 implicit. The remaining 1,407 videos are non-hate controls.
- Audio is the most common contributing modality in the TikTok hateful subset. This matters because text-only moderation would miss many examples.
- Target-group labels are multi-label. Adding the target columns does not produce a count of unique hateful videos.
- There are no comment fields in the public workbook. Mean, median, or distribution of comments per video therefore cannot be calculated from DeHate.